# Combined Client Value Storyboard

## Main business question

**Where is client value created, what explains the client experience, and which relationships should management protect or improve?**

This notebook combines the richer narrative structure from `storyboard_kayden(1).ipynb` with the triple-dashboard navigation from `storyboard.ipynb`. It presents one connected decision journey:

1. **Where is value created?** — identify profitability, margin quality and cost drivers.
2. **What explains the experience?** — compare service ratings, observed NPS ratings and service dimensions associated with advocacy.
3. **What should management do?** — connect client-level gross profit, satisfaction and advocacy to relative action priorities.

The dashboard uses `merged_cleaned.xlsx`, with one row per client-year observation. The first two sections retain the client-year grain. The advocacy section aggregates the selected observed years to one record per client before assigning client-level advocacy categories and action zones. All findings are descriptive associations; observed longevity is historical experience rather than a validated retention or churn probability.


In [1]:
# 1. Imports, data preparation and validation
# Load dependencies, define shared constants, and validate the cleaned workbook.
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, Input, Output, State, callback_context, dcc, html


DATA_PATH = Path("merged_cleaned.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Place merged_cleaned.xlsx in the same folder as this notebook, "
        "then run the cells again."
    )

# Load the workbook and normalise column names.
raw = pd.read_excel(DATA_PATH)
raw.columns = raw.columns.astype(str).str.strip()

client_id_candidates = [
    "Client ID", "CLIENT ID", "ClientID", "CLIENTID",
    "client_id", "Client_Id", "Client Id"
]
client_id_column = next(
    (column for column in client_id_candidates if column in raw.columns),
    None
)
if client_id_column is None:
    raise KeyError(
        f"No client ID column was found. Checked: {client_id_candidates}"
    )

raw = raw.rename(columns={client_id_column: "Client ID"})

SERVICE_COLUMNS = [
    "PRESALES AND PARTNERSHIP",
    "TECHNICAL EXPERTISE",
    "PROJECT DELIVERY",
    "POST-SALES SUPPORT",
]

SERVICE_LABELS = {
    "PRESALES AND PARTNERSHIP": "Presales & Partnership",
    "TECHNICAL EXPERTISE": "Technical Expertise",
    "PROJECT DELIVERY": "Project Delivery",
    "POST-SALES SUPPORT": "Post-Sales Support",
}

SERVICE_COLOURS = {
    "Presales & Partnership": "#636EFA",
    "Technical Expertise": "#EF553B",
    "Project Delivery": "#00CC96",
    "Post-Sales Support": "#AB63FA",
}

NPS_ORDER = ["Promoter", "Passive", "Detractor"]
NPS_COLOURS = {
    "Promoter": "#168AAD",
    "Passive": "#F4A261",
    "Detractor": "#E76F51",
}

SEGMENT_LABELS = {
    "TYPE": "Client Type",
    "SECTOR": "Industry Sector",
    "STAFF STRENGTH": "Organisation Size",
    "COUNTRY": "Country",
}

CLIENT_PROFILE_MAP = {
    "TYPE": "Client_Type",
    "SECTOR": "Sector",
    "STAFF STRENGTH": "Organisation_Size",
    "COUNTRY": "Country",
}

required_columns = {
    "Client ID", "YEAR", "TYPE", "SECTOR", "STAFF STRENGTH",
    "COUNTRY", "REVENUE", "HARDWARE", "SOFTWARE", "MANPOWER",
    "NPS RATING", *SERVICE_COLUMNS,
}
missing_columns = sorted(required_columns - set(raw.columns))
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

numeric_columns = [
    "YEAR", "REVENUE", "HARDWARE", "SOFTWARE", "MANPOWER",
    "NPS RATING", *SERVICE_COLUMNS,
]
for column in numeric_columns:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

# Derive record-level satisfaction, cost, profit, margin, and NPS fields.
analysis = raw.copy()
analysis["YEAR"] = analysis["YEAR"].astype("Int64")
analysis["Overall Satisfaction"] = analysis[SERVICE_COLUMNS].mean(axis=1)
analysis["COGS"] = analysis[[
    "HARDWARE", "SOFTWARE", "MANPOWER"
]].sum(axis=1, min_count=3)
analysis["Gross Profit"] = analysis["REVENUE"] - analysis["COGS"]
analysis["Gross Margin"] = (
    analysis["Gross Profit"]
    / analysis["REVENUE"].replace(0, np.nan)
)
analysis["NPS Category"] = np.select(
    [analysis["NPS RATING"].ge(9), analysis["NPS RATING"].ge(7)],
    ["Promoter", "Passive"],
    default="Detractor",
)
analysis["NPS Category"] = pd.Categorical(
    analysis["NPS Category"], categories=NPS_ORDER, ordered=True
)

# Build filter options and confirm the expected client-year grain.
years = sorted(analysis["YEAR"].dropna().astype(int).unique().tolist())
if not years:
    raise ValueError("YEAR contains no valid values.")

year_min, year_max = min(years), max(years)
client_types = sorted(analysis["TYPE"].dropna().astype(str).unique())
countries = sorted(analysis["COUNTRY"].dropna().astype(str).unique())

duplicate_client_years = int(
    analysis.duplicated(subset=["Client ID", "YEAR"]).sum()
)
if duplicate_client_years:
    raise ValueError(
        f"Found {duplicate_client_years} duplicate client-year records. "
        "Resolve them before using the storyboard."
    )

print(
    f"Loaded {len(analysis):,} client-year records, "
    f"{analysis['Client ID'].nunique():,} clients, "
    f"covering {year_min}–{year_max}."
)
print("Duplicate client-year records:", duplicate_client_years)


# Data grain note: client-year figures remain record-level; advocacy figures are client-level.


Loaded 392 client-year records, 140 clients, covering 2021–2025.
Duplicate client-year records: 0


In [2]:
# 2. Shared calculations, filters and dashboard components
# Define reusable filters, aggregations, styles, and dashboard card components.

GRAPH_CONFIG = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
    "toImageButtonOptions": {
        "format": "png",
        "filename": "client_value_storyboard",
        "scale": 2,
    },
}

PAGE_STYLE = {
    "fontFamily": "Segoe UI, Arial, sans-serif",
    "backgroundColor": "#F3F6FA",
    "color": "#172B4D",
    "minHeight": "100vh",
}

CONTENT_STYLE = {
    "maxWidth": "1500px",
    "margin": "0 auto",
    "padding": "0 22px 28px",
}

CARD_STYLE = {
    "backgroundColor": "white",
    "border": "1px solid #E3E8EF",
    "borderRadius": "12px",
    "boxShadow": "0 2px 8px rgba(23,43,77,0.06)",
}

# Return a consistent empty chart when a filter produces no data.
def empty_figure(title, message, height=520):
    figure = go.Figure()
    figure.add_annotation(
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        text=message,
        showarrow=False,
        font={"size": 16, "color": "#5E6C84"},
        align="center",
    )
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=height,
        xaxis={"visible": False},
        yaxis={"visible": False},
        margin={"t": 80, "r": 35, "b": 45, "l": 35},
    )
    return figure


# Apply the global year, client-type, country, and NPS filters.
def filter_rows(
    year_range,
    client_type="All",
    country="All",
    nps_categories=None,
):
    start_year, end_year = year_range
    filtered = analysis[
        analysis["YEAR"].between(start_year, end_year)
    ].copy()

    if client_type != "All":
        filtered = filtered[filtered["TYPE"].astype(str).eq(client_type)]
    if country != "All":
        filtered = filtered[
            filtered["COUNTRY"].astype(str).eq(country)
        ]

    if nps_categories is not None:
        if nps_categories:
            filtered = filtered[
                filtered["NPS Category"].astype(str).isin(nps_categories)
            ]
        else:
            filtered = filtered.iloc[0:0].copy()

    return filtered


# Aggregate selected client-year records for advocacy and priority analysis.
def build_client_level(filtered_rows):
    """Aggregate filtered client-year rows to one record per client."""
    columns = [
        "Client ID", "Client_Type", "Sector", "Organisation_Size",
        "Country", "Revenue", "Gross_Profit", "Gross_Margin",
        "Average_Satisfaction", "Average_NPS", "First_Observed_Year",
        "Last_Observed_Year", "Observed_Longevity", "Client_Year_Records",
        "NPS Category", "Priority Group",
    ]
    if filtered_rows.empty:
        return pd.DataFrame(columns=columns)

    client = (
        filtered_rows
        .groupby("Client ID", as_index=False, observed=True)
        .agg(
            Client_Type=("TYPE", "first"),
            Sector=("SECTOR", "first"),
            Organisation_Size=("STAFF STRENGTH", "first"),
            Country=("COUNTRY", "first"),
            Revenue=("REVENUE", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Average_Satisfaction=("Overall Satisfaction", "mean"),
            Average_NPS=("NPS RATING", "mean"),
            First_Observed_Year=("YEAR", "min"),
            Last_Observed_Year=("YEAR", "max"),
            Client_Year_Records=("YEAR", "size"),
        )
    )

    client["Gross_Margin"] = (
        client["Gross_Profit"]
        / client["Revenue"].replace(0, np.nan)
    )
    client["Observed_Longevity"] = (
        client["Last_Observed_Year"]
        - client["First_Observed_Year"]
        + 1
    )
    client["NPS Category"] = np.select(
        [client["Average_NPS"].ge(9), client["Average_NPS"].ge(7)],
        ["Promoter", "Passive"],
        default="Detractor",
    )
    client["NPS Category"] = pd.Categorical(
        client["NPS Category"], categories=NPS_ORDER, ordered=True
    )

    satisfaction_median = client["Average_Satisfaction"].median()
    margin_median = client["Gross_Margin"].median()
    client["Priority Group"] = np.select(
        [
            (
                client["Average_Satisfaction"].lt(satisfaction_median)
                & client["Gross_Margin"].ge(margin_median)
            ),
            (
                client["Average_Satisfaction"].ge(satisfaction_median)
                & client["Gross_Margin"].ge(margin_median)
            ),
            (
                client["Average_Satisfaction"].ge(satisfaction_median)
                & client["Gross_Margin"].lt(margin_median)
            ),
        ],
        ["Prioritise", "Retain", "Improve"],
        default="Reconsider",
    )
    return client[columns]


# Summarise financial, service, and advocacy measures by profile.
def profile_summary(filtered_rows, profile_column):
    if filtered_rows.empty:
        return pd.DataFrame()

    summary = (
        filtered_rows
        .groupby(profile_column, dropna=False, observed=True)
        .agg(
            Records=("Client ID", "size"),
            Clients=("Client ID", "nunique"),
            Revenue=("REVENUE", "sum"),
            Gross_Profit=("Gross Profit", "sum"),
            Average_Service=("Overall Satisfaction", "mean"),
            Average_NPS=("NPS RATING", "mean"),
        )
        .reset_index()
        .rename(columns={profile_column: "Profile"})
    )
    summary["Profile"] = summary["Profile"].fillna("Unknown").astype(str)
    summary["Gross_Margin"] = (
        summary["Gross_Profit"]
        / summary["Revenue"].replace(0, np.nan)
    )
    return summary


# Convert the active filters into a readable dashboard summary.
def filter_description(
    year_range,
    client_type,
    country,
    nps_categories,
):
    years_text = (
        str(year_range[0])
        if year_range[0] == year_range[1]
        else f"{year_range[0]}–{year_range[1]}"
    )
    type_text = "All client types" if client_type == "All" else client_type
    country_text = "All countries" if country == "All" else country
    nps_text = (
        "All advocacy groups"
        if set(nps_categories or []) == set(NPS_ORDER)
        else ", ".join(nps_categories or ["No advocacy groups"])
    )
    return f"{type_text} | {country_text} | {years_text} | {nps_text}"


# Build reusable KPI and evidence-card components for the layout.
def metric_card(label, value, context):
    return html.Div(
        [
            html.Div(
                label,
                style={
                    "fontSize": "12px",
                    "fontWeight": "700",
                    "color": "#5E6C84",
                    "textTransform": "uppercase",
                    "letterSpacing": "0.45px",
                },
            ),
            html.Div(
                value,
                style={
                    "fontSize": "24px",
                    "fontWeight": "750",
                    "color": "#172B4D",
                    "marginTop": "5px",
                },
            ),
            html.Div(
                context,
                style={
                    "fontSize": "12px",
                    "color": "#6B778C",
                    "marginTop": "5px",
                },
            ),
        ],
        style={**CARD_STYLE, "padding": "15px 17px"},
    )


def kpi_cards(filtered_rows):
    if filtered_rows.empty:
        values = [
            ("Gross profit", "—", "No matching records"),
            ("Weighted gross margin", "—", "No matching records"),
            ("Average satisfaction", "—", "Four service dimensions"),
            ("Average NPS rating", "—", "Individual 0–10 rating"),
        ]
    else:
        gross_profit = filtered_rows["Gross Profit"].sum()
        revenue = filtered_rows["REVENUE"].sum()
        gross_margin = gross_profit / revenue if revenue else np.nan
        values = [
            (
                "Gross profit",
                f"${gross_profit / 1_000_000:,.1f}M",
                f"Revenue ${revenue / 1_000_000:,.1f}M",
            ),
            (
                "Weighted gross margin",
                f"{gross_margin:.1%}" if pd.notna(gross_margin) else "—",
                "Total profit ÷ total revenue",
            ),
            (
                "Average satisfaction",
                f"{filtered_rows['Overall Satisfaction'].mean():.2f} / 5",
                "Mean of four service ratings",
            ),
            (
                "Average NPS rating",
                f"{filtered_rows['NPS RATING'].mean():.2f} / 10",
                (
                    f"{len(filtered_rows):,} records | "
                    f"{filtered_rows['Client ID'].nunique():,} clients"
                ),
            ),
        ]
    return [metric_card(*item) for item in values]


# Wrap each figure with an evidence label, question, and interpretation guide.
def graph_card(graph_id, evidence_label, question, guide, height=600):
    return html.Div(
        [
            html.Div(
                evidence_label,
                style={
                    "fontSize": "12px",
                    "fontWeight": "800",
                    "letterSpacing": "0.8px",
                    "color": "#0F6B78",
                    "textTransform": "uppercase",
                },
            ),
            html.H3(
                question,
                style={
                    "margin": "4px 0 5px",
                    "fontSize": "21px",
                    "color": "#172B4D",
                },
            ),
            html.P(
                guide,
                style={
                    "margin": "0 0 4px",
                    "fontSize": "13px",
                    "lineHeight": "1.45",
                    "color": "#5E6C84",
                },
            ),
            dcc.Graph(
                id=graph_id,
                config=GRAPH_CONFIG,
                style={"height": f"{height}px", "width": "100%"},
            ),
        ],
        style={**CARD_STYLE, "padding": "18px 20px 8px", "minWidth": 0},
    )


# Create a short handoff message between dashboard sections.
def transition_card(title, text):
    return html.Div(
        [
            html.Div(
                title,
                style={
                    "fontWeight": "800",
                    "color": "#0F6B78",
                    "marginBottom": "5px",
                },
            ),
            html.Div(
                text,
                style={"color": "#425466", "lineHeight": "1.55"},
            ),
        ],
        style={
            "backgroundColor": "#E9F5F5",
            "borderLeft": "5px solid #2A9D8F",
            "padding": "15px 18px",
            "borderRadius": "8px",
        },
    )


In [3]:
# 3. Profitability and cost-driver figures
# Build the profitability, cost-composition, and value-experience figures.

# Compare gross profit, revenue, or client count by selected profile.
def profit_figure(filtered_rows, profile_column, metric="Gross_Profit"):
    metric_labels = {"Gross_Profit": "Gross profit", "Revenue": "Revenue", "Clients": "Client count"}
    title = "Which client profiles create the most value?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")
    summary = profile_summary(filtered_rows, profile_column)
    summary = summary.sort_values(metric, ascending=True)
    metric_label = metric_labels[metric]
    is_currency = metric != "Clients"
    chart_height = max(520, min(820, 42 * len(summary) + 190))
    figure = px.bar(
        summary, x=metric, y="Profile", orientation="h",
        color=summary["Gross_Margin"] * 100, color_continuous_scale="Viridis",
        text=metric,
        custom_data=["Profile", "Revenue", "Gross_Profit", "Gross_Margin", "Clients", "Records", "Average_Service", "Average_NPS"],
        title=f"{metric_label} by selected profile",
    )
    figure.update_traces(
        texttemplate="$%{x:,.0f}" if is_currency else "%{x:,.0f}",
        textposition="outside", cliponaxis=False,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>" + metric_label + (": $%{x:,.0f}<br>" if is_currency else ": %{x:,.0f}<br>") +
            "Gross profit: $%{customdata[2]:,.0f}<br>Revenue: $%{customdata[1]:,.0f}<br>"+
            "Weighted gross margin: %{customdata[3]:.1%}<br>Average satisfaction: %{customdata[6]:.2f}/5<br>"+
            "Average NPS rating: %{customdata[7]:.2f}/10<br>Clients: %{customdata[4]}<br>"+
            "Client-year records: %{customdata[5]}<extra></extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white", height=chart_height,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title=f"{metric_label} ({'$' if is_currency else 'count'})",
        yaxis_title=SEGMENT_LABELS[profile_column],
        coloraxis_colorbar={"title": "Weighted<br>gross margin (%)"},
        margin={"t": 80, "r": 100, "b": 65, "l": 150},
    )
    figure.update_xaxes(tickformat="$,.0f" if is_currency else ",.0f", rangemode="tozero")
    return figure

# Show hardware, software, and manpower costs by selected profile.
def cost_figure(filtered_rows, profile_column, display_mode):
    title = "Which costs are absorbing value within each profile?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    costs = (
        filtered_rows
        .groupby(profile_column, dropna=False, observed=True)[
            ["HARDWARE", "SOFTWARE", "MANPOWER"]
        ]
        .sum()
        .reset_index()
        .rename(columns={profile_column: "Profile"})
    )
    costs["Profile"] = costs["Profile"].fillna("Unknown").astype(str)
    costs["Total Cost"] = costs[
        ["HARDWARE", "SOFTWARE", "MANPOWER"]
    ].sum(axis=1)
    costs = costs.sort_values("Total Cost", ascending=True)
    chart_height = max(520, min(820, 42 * len(costs) + 190))

    cost_colours = {
        "HARDWARE": "#636EFA",
        "SOFTWARE": "#EF553B",
        "MANPOWER": "#00CC96",
    }
    figure = go.Figure()
    for column in ["HARDWARE", "SOFTWARE", "MANPOWER"]:
        figure.add_trace(
            go.Bar(
                x=costs[column],
                y=costs["Profile"],
                orientation="h",
                name=column.title(),
                marker_color=cost_colours[column],
                customdata=costs[["Total Cost"]].to_numpy(),
                hovertemplate=(
                    f"<b>%{{y}}</b><br>{column.title()}: $%{{x:,.0f}}<br>"
                    "Total cost: $%{customdata[0]:,.0f}<extra></extra>"
                ),
            )
        )

    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=chart_height,
        barmode=display_mode,
        xaxis_title="Total cost ($)",
        yaxis_title=SEGMENT_LABELS[profile_column],
        legend={
            "title": "Cost component",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 110, "r": 40, "b": 65, "l": 150},
    )
    figure.update_xaxes(tickformat="$,.0f", rangemode="tozero")
    return figure


# Relate profile-level service satisfaction to weighted gross margin.
def value_experience_figure(filtered_rows, profile_column):
    title = "Which profiles combine healthy margins with strong service satisfaction?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    summary = profile_summary(filtered_rows, profile_column)
    if summary.empty:
        return empty_figure(title, "No profile summary can be calculated.")

    summary["Margin_Percent"] = summary["Gross_Margin"] * 100
    summary["Bubble_Value"] = summary["Revenue"].clip(lower=0)

    figure = px.scatter(
        summary,
        x="Average_Service",
        y="Margin_Percent",
        size="Bubble_Value",
        size_max=58,
        color="Gross_Profit",
        color_continuous_scale="Viridis",
        text="Profile",
        custom_data=[
            "Profile", "Gross_Profit", "Revenue", "Average_NPS",
            "Clients", "Records",
        ],
        title=title,
    )
    figure.update_traces(
        textposition="top center",
        marker={"line": {"width": 1, "color": "white"}, "opacity": 0.84},
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Average satisfaction: %{x:.2f}/5<br>"
            "Weighted gross margin: %{y:.1f}%<br>"
            "Gross profit: $%{customdata[1]:,.0f}<br>"
            "Revenue: $%{customdata[2]:,.0f}<br>"
            "Average NPS rating: %{customdata[3]:.2f}/10<br>"
            "Clients: %{customdata[4]}<br>"
            "Client-year records: %{customdata[5]}"
            "<extra></extra>"
        ),
    )

    satisfaction_median = summary["Average_Service"].median()
    margin_median = summary["Margin_Percent"].median()
    figure.add_vline(
        x=satisfaction_median,
        line_dash="dash",
        line_color="#5E6C84",
        annotation_text=f"Profile median {satisfaction_median:.2f}",
        annotation_position="top left",
    )
    figure.add_hline(
        y=margin_median,
        line_dash="dash",
        line_color="#5E6C84",
        annotation_text=f"Profile median {margin_median:.1f}%",
        annotation_position="bottom right",
    )
    figure.update_layout(
        template="plotly_white",
        height=610,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Average service satisfaction (1–5)",
        yaxis_title="Weighted gross margin (%)",
        coloraxis_colorbar={"title": "Gross profit ($)"},
        margin={"t": 85, "r": 90, "b": 70, "l": 80},
    )
    figure.update_xaxes(range=[1, 5.15])
    return figure


In [4]:
# 4. Satisfaction and loyalty figures
# Build service-performance, NPS-trend, and service-priority figures.

# Compare selected service ratings by client type and period.
def service_figure(filtered_rows, selected_services, selected_period):
    title = "Where are service strengths and gaps?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")
    if not selected_services:
        return empty_figure(
            title, "Select at least one service dimension in the filter bar."
        )

    if selected_period == "Overall":
        chart_rows = filtered_rows.copy()
        period_label = "Filtered period overall"
    else:
        chart_rows = filtered_rows[
            filtered_rows["YEAR"].eq(int(selected_period))
        ].copy()
        period_label = str(selected_period)

    if chart_rows.empty:
        return empty_figure(
            title,
            "The selected service period is outside the current year filter.",
        )

    service_summary = (
        chart_rows
        .groupby("TYPE", observed=True)[selected_services]
        .mean()
        .reset_index()
    )
    counts = chart_rows.groupby("TYPE", observed=True).size()
    service_long = service_summary.melt(
        id_vars="TYPE",
        value_vars=selected_services,
        var_name="Service Column",
        value_name="Average Rating",
    )
    service_long["Service Dimension"] = service_long[
        "Service Column"
    ].map(SERVICE_LABELS)
    service_long["Records"] = service_long["TYPE"].map(counts)

    figure = px.bar(
        service_long,
        x="TYPE",
        y="Average Rating",
        color="Service Dimension",
        barmode="group",
        text="Average Rating",
        custom_data=["Records"],
        color_discrete_map=SERVICE_COLOURS,
        category_orders={
            "TYPE": client_types,
            "Service Dimension": [
                SERVICE_LABELS[column] for column in selected_services
            ],
        },
        title=f"{title} — {period_label}",
    )
    figure.update_traces(
        texttemplate="%{y:.2f}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "Client type: %{x}<br>"
            "Average rating: %{y:.2f}/5<br>"
            "Client-year records: %{customdata[0]}"
            "<extra>%{fullData.name}</extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white",
        height=590,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Client type",
        yaxis={"title": "Average rating (1–5)", "range": [0, 5.35]},
        legend={
            "title": "Service dimension",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 120, "r": 35, "b": 65, "l": 70},
    )
    return figure


# Display average NPS ratings by client type and observed year.
def nps_heatmap_figure(filtered_rows):
    title = "Which client types show the strongest customer advocacy?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")

    grouped = (
        filtered_rows
        .groupby(["TYPE", "YEAR"], observed=True)["NPS RATING"]
        .agg(["mean", "count"])
        .reset_index()
    )
    values = grouped.pivot(index="TYPE", columns="YEAR", values="mean")
    counts = grouped.pivot(index="TYPE", columns="YEAR", values="count")

    displayed_types = [
        item for item in client_types if item in values.index
    ]
    displayed_years = [item for item in years if item in values.columns]

    values = values.reindex(index=displayed_types, columns=displayed_years)
    counts = counts.reindex(index=displayed_types, columns=displayed_years)
    values["Overall"] = (
        filtered_rows.groupby("TYPE", observed=True)["NPS RATING"]
        .mean()
        .reindex(displayed_types)
    )
    counts["Overall"] = (
        filtered_rows.groupby("TYPE", observed=True).size()
        .reindex(displayed_types)
    )

    heatmap_text = np.array([
        ["" if pd.isna(value) else f"{value:.2f}" for value in row]
        for row in values.to_numpy()
    ])

    figure = go.Figure(
        go.Heatmap(
            z=values.to_numpy(),
            x=[str(column) for column in values.columns],
            y=values.index.tolist(),
            text=heatmap_text,
            texttemplate="%{text}",
            textfont={"size": 16},
            customdata=counts.to_numpy(),
            zmin=7,
            zmax=9,
            colorscale=[
                [0.00, "#E76F51"],
                [0.50, "#F4D35E"],
                [1.00, "#2A9D8F"],
            ],
            colorbar={"title": "Average<br>NPS rating"},
            hovertemplate=(
                "Client type: %{y}<br>"
                "Period: %{x}<br>"
                "Average NPS rating: %{z:.2f}/10<br>"
                "Client-year records: %{customdata:.0f}"
                "<extra></extra>"
            ),
        )
    )
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=560,
        xaxis_title="Period",
        yaxis_title="Client type",
        margin={"t": 85, "r": 90, "b": 70, "l": 85},
    )
    return figure


# Calculate a rank correlation without requiring SciPy.
def spearman_rank_correlation(left, right):
    """Calculate Spearman correlation without requiring SciPy."""
    pair = pd.concat([left, right], axis=1).dropna()
    if len(pair) < 3 or pair.iloc[:, 0].nunique() < 2 or pair.iloc[:, 1].nunique() < 2:
        return np.nan
    ranked = pair.rank(method="average")
    correlation = np.corrcoef(ranked.iloc[:, 0], ranked.iloc[:, 1])[0, 1]
    return float(correlation) if np.isfinite(correlation) else np.nan


# Score service dimensions by performance and association with NPS.
def service_priority_data(filtered_rows, selected_services):
    rows = []
    for column in selected_services:
        pair = filtered_rows[[column, "NPS RATING"]].dropna()
        enough_variation = (
            len(pair) >= 3
            and pair[column].nunique() >= 2
            and pair["NPS RATING"].nunique() >= 2
        )
        correlation = (
            spearman_rank_correlation(pair[column], pair["NPS RATING"])
            if enough_variation else np.nan
        )
        rows.append({
            "Service Column": column,
            "Service Dimension": SERVICE_LABELS[column],
            "Performance": float(pair[column].mean()) if len(pair) else np.nan,
            "Importance": float(correlation),
            "Records": len(pair),
        })

    priority = pd.DataFrame(rows).dropna(
        subset=["Performance", "Importance"]
    )
    if len(priority) < 2:
        return priority

    performance_cutoff = float(priority["Performance"].median())
    importance_cutoff = float(priority["Importance"].median())

    def assign_action(row):
        high_performance = row["Performance"] >= performance_cutoff
        high_importance = row["Importance"] >= importance_cutoff
        if not high_performance and high_importance:
            return "Improve First"
        if high_performance and high_importance:
            return "Protect Strength"
        if not high_performance and not high_importance:
            return "Monitor"
        return "Maintain"

    priority["Recommended Action"] = priority.apply(assign_action, axis=1)
    return priority


# Plot relative service action zones from the calculated priorities.
def service_priority_figure(filtered_rows, selected_services, segment_label):
    title = "What should be improved or protected first?"
    if filtered_rows.empty:
        return empty_figure(title, "No data match the selected filters.")
    if len(selected_services or []) < 2:
        return empty_figure(
            title,
            "Select at least two service dimensions to create relative priorities.",
        )

    priority = service_priority_data(filtered_rows, selected_services)
    if len(priority) < 2:
        return empty_figure(
            title,
            "The selected data do not contain enough variation for correlations.",
        )

    performance_cutoff = float(priority["Performance"].median())
    importance_cutoff = float(priority["Importance"].median())
    x_span = float(priority["Performance"].max() - priority["Performance"].min())
    y_span = float(priority["Importance"].max() - priority["Importance"].min())
    x_padding = max(0.16, x_span * 0.42)
    y_padding = max(0.08, y_span * 0.42)
    x_min = max(1.0, float(priority["Performance"].min() - x_padding))
    x_max = min(5.0, float(priority["Performance"].max() + x_padding))
    y_min = max(-1.0, float(priority["Importance"].min() - y_padding))
    y_max = min(1.0, float(priority["Importance"].max() + y_padding))
    if x_max - x_min < 0.35:
        midpoint = (x_min + x_max) / 2
        x_min, x_max = max(1, midpoint - 0.22), min(5, midpoint + 0.22)
    if y_max - y_min < 0.25:
        midpoint = (y_min + y_max) / 2
        y_min, y_max = max(-1, midpoint - 0.14), min(1, midpoint + 0.14)

    action_colours = {
        "Improve First": "#E76F51",
        "Protect Strength": "#2A9D8F",
        "Monitor": "#F4A261",
        "Maintain": "#457B9D",
    }
    text_positions = {
        "Presales & Partnership": "bottom left",
        "Technical Expertise": "top left",
        "Project Delivery": "top right",
        "Post-Sales Support": "bottom right",
    }

    figure = go.Figure()
    quadrants = [
        (x_min, performance_cutoff, importance_cutoff, y_max,
         "#FDE7E2", "Improve First", x_min, y_max, "left", "top"),
        (performance_cutoff, x_max, importance_cutoff, y_max,
         "#DDF2EC", "Protect Strength", x_max, y_max, "right", "top"),
        (x_min, performance_cutoff, y_min, importance_cutoff,
         "#FFF1DB", "Monitor", x_min, y_min, "left", "bottom"),
        (performance_cutoff, x_max, y_min, importance_cutoff,
         "#E6EEF7", "Maintain", x_max, y_min, "right", "bottom"),
    ]
    for x0, x1, y0, y1, colour, label, lx, ly, xa, ya in quadrants:
        figure.add_shape(
            type="rect", x0=x0, x1=x1, y0=y0, y1=y1,
            fillcolor=colour, opacity=0.62, line_width=0, layer="below",
        )
        figure.add_annotation(
            x=lx, y=ly, text=f"<b>{label}</b>", showarrow=False,
            xanchor=xa, yanchor=ya,
            xshift=10 if xa == "left" else -10,
            yshift=-9 if ya == "top" else 9,
            font={"size": 12, "color": "#42526E"},
        )

    for action in [
        "Improve First", "Protect Strength", "Monitor", "Maintain"
    ]:
        action_rows = priority[
            priority["Recommended Action"].eq(action)
        ]
        if action_rows.empty:
            continue
        figure.add_trace(
            go.Scatter(
                x=action_rows["Performance"],
                y=action_rows["Importance"],
                mode="markers+text",
                name=action,
                text=action_rows["Service Dimension"],
                textposition=[
                    text_positions.get(item, "top center")
                    for item in action_rows["Service Dimension"]
                ],
                marker={
                    "size": 18,
                    "color": action_colours[action],
                    "line": {"width": 2, "color": "white"},
                },
                customdata=action_rows[
                    ["Records", "Recommended Action"]
                ].to_numpy(dtype=object),
                hovertemplate=(
                    "<b>%{text}</b><br>"
                    "Average rating: %{x:.3f}/5<br>"
                    "Spearman correlation: %{y:+.3f}<br>"
                    "Valid records: %{customdata[0]}<br>"
                    "Recommended action: %{customdata[1]}"
                    "<extra></extra>"
                ),
            )
        )

    figure.add_vline(
        x=performance_cutoff, line_dash="dash",
        line_color="#5E6C84", line_width=2,
    )
    figure.add_hline(
        y=importance_cutoff, line_dash="dash",
        line_color="#5E6C84", line_width=2,
    )
    figure.update_layout(
        title={
            "text": (
                f"{title} — {segment_label}<br><sup>"
                "Performance = average rating; importance = association with NPS"
                "</sup>"
            ),
            "x": 0.5,
            "xanchor": "center",
        },
        template="plotly_white",
        height=620,
        xaxis={
            "title": "Current performance — average service rating (1–5)",
            "range": [x_min, x_max],
            "tickformat": ".2f",
            "nticks": 6,
        },
        yaxis={
            "title": "Relative importance — Spearman correlation with NPS",
            "range": [y_min, y_max],
            "tickformat": ".2f",
            "nticks": 6,
        },
        legend={
            "title": "Recommended action",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 125, "r": 55, "b": 80, "l": 90},
    )
    return figure


In [5]:
# 5. Advocacy and client-prioritisation figures
# Build advocacy-value, portfolio-composition, and client-priority figures.

# Compare gross-profit contribution across client advocacy categories.
def advocacy_profit_figure(client_rows, profile_column):
    title = "Where is commercial value concentrated across advocacy groups?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    base = client_rows.copy()
    base["Profile"] = base[profile_column].fillna("Unknown").astype(str)
    grouped = (
        base
        .groupby(["Profile", "NPS Category"], observed=True)
        .agg(
            Gross_Profit=("Gross_Profit", "sum"),
            Revenue=("Revenue", "sum"),
            Clients=("Client ID", "size"),
            Median_Longevity=("Observed_Longevity", "median"),
            Average_Satisfaction=("Average_Satisfaction", "mean"),
        )
        .reset_index()
    )
    grouped["Gross_Margin"] = (
        grouped["Gross_Profit"]
        / grouped["Revenue"].replace(0, np.nan)
    )
    grouped["NPS Category"] = pd.Categorical(
        grouped["NPS Category"], categories=NPS_ORDER, ordered=True
    )
    grouped = grouped.sort_values(
        ["NPS Category", "Gross_Profit"], ascending=[False, True]
    )
    grouped["Label"] = (
        grouped["Profile"].astype(str)
        + " — "
        + grouped["NPS Category"].astype(str)
    )
    grouped["Value Label"] = grouped["Gross_Profit"].map(
        lambda value: f"${value / 1_000_000:.1f}M"
    )
    chart_height = max(560, min(900, 35 * len(grouped) + 210))

    figure = px.bar(
        grouped,
        x="Gross_Profit",
        y="Label",
        orientation="h",
        color="NPS Category",
        text="Value Label",
        color_discrete_map=NPS_COLOURS,
        category_orders={"NPS Category": NPS_ORDER},
        custom_data=[
            "Profile", "NPS Category", "Revenue", "Gross_Profit",
            "Gross_Margin", "Clients", "Median_Longevity",
            "Average_Satisfaction",
        ],
        title=title,
    )
    figure.update_traces(
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{customdata[0]} — %{customdata[1]}</b><br>"
            "Gross profit: $%{customdata[3]:,.0f}<br>"
            "Revenue: $%{customdata[2]:,.0f}<br>"
            "Weighted gross margin: %{customdata[4]:.1%}<br>"
            "Clients: %{customdata[5]}<br>"
            "Median observed longevity: %{customdata[6]:.1f} years<br>"
            "Average satisfaction: %{customdata[7]:.2f}/5"
            "<extra></extra>"
        ),
    )
    figure.update_layout(
        template="plotly_white",
        height=chart_height,
        title={"x": 0.5, "xanchor": "center"},
        xaxis_title="Total gross profit ($)",
        yaxis_title="",
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 55, "b": 70, "l": 210},
    )
    figure.update_xaxes(tickformat="$,.0f", rangemode="tozero")
    return figure


# Show the Promoter, Passive, and Detractor mix within each profile.
def advocacy_composition_figure(client_rows, profile_column):
    title = "Which profiles contain Promoters, Passives and Detractors?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    base = client_rows.copy()
    base["Profile"] = base[profile_column].fillna("Unknown").astype(str)
    summary = (
        base.groupby("Profile", observed=True)
        .agg(
            Clients=("Client ID", "size"),
            Revenue=("Revenue", "sum"),
            Gross_Profit=("Gross_Profit", "sum"),
            Median_Longevity=("Observed_Longevity", "median"),
        )
    )
    counts = (
        base.groupby(["Profile", "NPS Category"], observed=True)
        .size()
        .unstack(fill_value=0)
    )
    counts.columns = counts.columns.astype(str)
    for category in NPS_ORDER:
        if category not in counts.columns:
            counts[category] = 0
    composition = summary.join(counts[NPS_ORDER], how="left")
    composition = composition.sort_values("Gross_Profit", ascending=True)

    figure = go.Figure()
    for category in NPS_ORDER:
        share = composition[category] / composition["Clients"] * 100
        customdata = np.column_stack([
            composition.index.to_numpy(),
            composition["Clients"].to_numpy(),
            composition["Gross_Profit"].to_numpy(),
            composition["Revenue"].to_numpy(),
            composition["Median_Longevity"].to_numpy(),
            composition[category].to_numpy(),
        ])
        figure.add_trace(
            go.Bar(
                x=share,
                y=composition.index,
                orientation="h",
                name=category,
                marker_color=NPS_COLOURS[category],
                text=[f"{value:.0f}%" if value >= 7 else "" for value in share],
                textposition="inside",
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    f"Advocacy category: {category}<br>"
                    "Share: %{x:.1f}%<br>"
                    "Clients in category: %{customdata[5]:.0f}<br>"
                    "Profile gross profit: $%{customdata[2]:,.0f}<br>"
                    "Total clients: %{customdata[1]:.0f}<br>"
                    "Median observed longevity: %{customdata[4]:.1f} years"
                    "<extra></extra>"
                ),
            )
        )

    annotations = []
    for profile, row in composition.iterrows():
        annotations.append(
            {
                "x": 101.5,
                "y": profile,
                "xref": "x",
                "yref": "y",
                "text": (
                    f"GP ${row['Gross_Profit'] / 1_000_000:.1f}M | "
                    f"n={int(row['Clients'])} | "
                    f"{row['Median_Longevity']:.1f}y"
                ),
                "showarrow": False,
                "xanchor": "left",
                "font": {"size": 11, "color": "#5E6C84"},
            }
        )

    chart_height = max(560, min(900, 42 * len(composition) + 220))
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=chart_height,
        barmode="stack",
        xaxis={
            "title": "Share of clients within profile (%)",
            "range": [0, 123],
            "ticksuffix": "%",
        },
        yaxis={
            "title": "",
            "categoryorder": "array",
            "categoryarray": composition.index.tolist(),
        },
        annotations=annotations,
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 210, "b": 70, "l": 150},
    )
    return figure


# Plot account-level satisfaction, margin, profit, and advocacy priorities.
def client_priority_figure(client_rows):
    title = "Which clients should be retained, improved or reconsidered?"
    if client_rows.empty:
        return empty_figure(title, "No clients match the selected filters.")

    client_rows = client_rows.dropna(
        subset=["Average_Satisfaction", "Gross_Margin"]
    ).copy()
    if client_rows.empty:
        return empty_figure(title, "No valid satisfaction and margin pairs remain.")

    satisfaction_median = client_rows["Average_Satisfaction"].median()
    margin_median = client_rows["Gross_Margin"].median() * 100
    figure = go.Figure()

    for category in NPS_ORDER:
        category_rows = client_rows[
            client_rows["NPS Category"].astype(str).eq(category)
        ]
        if category_rows.empty:
            continue
        positive_profit = category_rows["Gross_Profit"].clip(lower=0)
        if positive_profit.max() > 0:
            marker_sizes = 10 + 30 * np.sqrt(
                positive_profit / positive_profit.max()
            )
        else:
            marker_sizes = np.full(len(category_rows), 12.0)

        customdata = category_rows[[
            "Client ID", "Client_Type", "Sector", "Country",
            "Revenue", "Gross_Profit", "Observed_Longevity",
            "Average_NPS", "Priority Group",
        ]].to_numpy(dtype=object)
        figure.add_trace(
            go.Scatter(
                x=category_rows["Average_Satisfaction"],
                y=category_rows["Gross_Margin"] * 100,
                mode="markers",
                name=category,
                marker={
                    "size": marker_sizes,
                    "color": NPS_COLOURS[category],
                    "opacity": 0.78,
                    "line": {"width": 0.9, "color": "white"},
                },
                customdata=customdata,
                hovertemplate=(
                    "<b>Client %{customdata[0]}</b><br>"
                    "Client type: %{customdata[1]}<br>"
                    "Sector: %{customdata[2]}<br>"
                    "Country: %{customdata[3]}<br>"
                    "Average satisfaction: %{x:.2f}/5<br>"
                    "Client-level gross margin: %{y:.1f}%<br>"
                    "Gross profit: $%{customdata[5]:,.0f}<br>"
                    "Average NPS: %{customdata[7]:.2f}/10<br>"
                    "Observed longevity: %{customdata[6]} years<br>"
                    "Action zone: %{customdata[8]}"
                    "<extra></extra>"
                ),
            )
        )

    figure.add_vline(
        x=satisfaction_median,
        line_dash="dash",
        line_color="#425466",
        line_width=2,
        annotation_text=f"Median satisfaction {satisfaction_median:.2f}",
        annotation_position="top right",
    )
    figure.add_hline(
        y=margin_median,
        line_dash="dash",
        line_color="#425466",
        line_width=2,
        annotation_text=f"Median margin {margin_median:.1f}%",
        annotation_position="bottom right",
    )

    x_min = max(1, client_rows["Average_Satisfaction"].min() - 0.15)
    x_max = min(5, client_rows["Average_Satisfaction"].max() + 0.15)
    y_values = client_rows["Gross_Margin"] * 100
    y_padding = max(4, (y_values.max() - y_values.min()) * 0.08)
    y_min, y_max = y_values.min() - y_padding, y_values.max() + y_padding
    quadrant_labels = [
        (x_min, y_max, "Prioritise", "left", "top", "#C23B31"),
        (x_max, y_max, "Retain", "right", "top", "#237A45"),
        (x_min, y_min, "Reconsider", "left", "bottom", "#7A4DD8"),
        (x_max, y_min, "Improve", "right", "bottom", "#2F6DB3"),
    ]
    for x, y, label, xa, ya, colour in quadrant_labels:
        figure.add_annotation(
            x=x, y=y, text=f"<b>{label}</b>", showarrow=False,
            xanchor=xa, yanchor=ya,
            xshift=10 if xa == "left" else -10,
            yshift=-10 if ya == "top" else 10,
            bgcolor="rgba(255,255,255,0.82)",
            bordercolor=colour,
            borderwidth=1,
            font={"size": 15, "color": colour},
        )

    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=680,
        xaxis={
            "title": "Average overall satisfaction (1–5)",
            "range": [x_min, x_max],
        },
        yaxis={
            "title": "Client-level gross margin (%)",
            "range": [y_min, y_max],
        },
        legend={
            "title": "Client-level advocacy category",
            "orientation": "h",
            "yanchor": "bottom",
            "y": 1.02,
            "xanchor": "center",
            "x": 0.5,
        },
        margin={"t": 115, "r": 50, "b": 75, "l": 90},
    )
    return figure


In [6]:
# 6. Story text, dashboard layout and navigation
# Define narrative cards, the three dashboard tabs, and shared navigation.

# Summarise the evidence currently selected by the user.
def evidence_summary(filtered_rows):
    if filtered_rows.empty:
        return html.Div("No records match the selected filters.")

    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    service_leader = type_summary.loc[type_summary["Average_Service"].idxmax()]
    nps_leader = type_summary.loc[type_summary["Average_NPS"].idxmax()]

    return html.Div(
        [
            html.Strong("Current evidence: "),
            html.Span(
                f"{profit_leader['Profile']} leads total gross profit "
                f"(${profit_leader['Gross_Profit'] / 1_000_000:.1f}M); "
                f"{service_leader['Profile']} has the strongest average "
                f"service satisfaction ({service_leader['Average_Service']:.2f}/5); "
                f"and {nps_leader['Profile']} has the highest average NPS "
                f"rating ({nps_leader['Average_NPS']:.2f}/10)."
            ),
        ],
        style={
            "backgroundColor": "#FFF9E8",
            "borderLeft": "5px solid #E6A700",
            "padding": "14px 18px",
            "borderRadius": "8px",
            "lineHeight": "1.55",
            "color": "#425466",
        },
    )


# Generate the profitability and experience conclusion for the active filters.
def main_conclusion(filtered_rows):
    if filtered_rows.empty:
        return html.Div("No conclusion can be calculated for an empty selection.")

    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    contribution_cutoff = type_summary["Gross_Profit"].median()
    financially_material = type_summary[
        type_summary["Gross_Profit"].ge(contribution_cutoff)
    ]
    balance_leader = financially_material.loc[
        financially_material["Average_Service"].idxmax()
    ]
    nps_leader = type_summary.loc[type_summary["Average_NPS"].idxmax()]

    leader_rows = filtered_rows[
        filtered_rows["TYPE"].astype(str).eq(str(profit_leader["Profile"]))
    ]
    service_diagnosis = service_priority_data(
        leader_rows, SERVICE_COLUMNS
    )
    performance_means = leader_rows[SERVICE_COLUMNS].mean()
    weakest_column = performance_means.idxmin()
    weakest_label = SERVICE_LABELS[weakest_column]
    if service_diagnosis.empty:
        associated_label = "the service dimensions with sufficient data"
    else:
        associated_label = service_diagnosis.loc[
            service_diagnosis["Importance"].idxmax(),
            "Service Dimension",
        ]

    if str(balance_leader["Profile"]) == str(profit_leader["Profile"]):
        balance_sentence = (
            f"{profit_leader['Profile']} also provides the strongest "
            "satisfaction balance among financially material client types."
        )
    else:
        balance_sentence = (
            f"{balance_leader['Profile']} provides the strongest balance "
            f"among client types at or above the median gross-profit "
            f"contribution, with satisfaction of "
            f"{balance_leader['Average_Service']:.2f}/5."
        )

    return html.Div(
        [
            html.H3(
                "Answer to the main question",
                style={"margin": "0 0 9px", "color": "#172B4D"},
            ),
            html.P(
                (
                    f"Within the current filters, {profit_leader['Profile']} "
                    f"generates the highest total gross profit at "
                    f"${profit_leader['Gross_Profit'] / 1_000_000:.1f}M, "
                    f"with average satisfaction of "
                    f"{profit_leader['Average_Service']:.2f}/5 and average "
                    f"NPS rating of {profit_leader['Average_NPS']:.2f}/10. "
                    f"{balance_sentence} {nps_leader['Profile']} records the "
                    f"highest advocacy level at {nps_leader['Average_NPS']:.2f}/10."
                ),
                style={"lineHeight": "1.6", "margin": "0 0 10px"},
            ),
            html.P(
                (
                    f"The commercial recommendation is to protect the "
                    f"high-value relationships while addressing "
                    f"{weakest_label}, the weakest service area for the "
                    f"gross-profit leader. At the same time, preserve "
                    f"{associated_label}, which has the strongest observed "
                    f"association with that segment's NPS rating. The final "
                    "client matrix should then be used to identify the "
                    "specific accounts to retain, improve, prioritise or reconsider."
                ),
                style={"lineHeight": "1.6", "margin": 0},
            ),
            html.Div(
                (
                    "Interpretation boundary: the dashboard identifies "
                    "descriptive patterns and associations. It does not prove "
                    "that a service change causes NPS, renewal or retention."
                ),
                style={
                    "fontSize": "12px",
                    "color": "#6B778C",
                    "marginTop": "11px",
                    "fontStyle": "italic",
                },
            ),
        ],
        style={
            **CARD_STYLE,
            "borderTop": "5px solid #2A9D8F",
            "padding": "20px 22px",
        },
    )



# Build a compact purpose-and-takeaway card for each section.
def purpose_card(question, purpose, takeaway):
    return html.Div(
        [
            html.Div("Section purpose", style={
                "fontSize": "11px", "fontWeight": "800", "letterSpacing": "1px",
                "textTransform": "uppercase", "color": "#0F6B78",
            }),
            html.Div([
                html.Strong("Question: "), html.Span(question),
            ], style={"marginTop": "5px", "lineHeight": "1.5"}),
            html.Div([
                html.Strong("Why it matters: "), html.Span(purpose),
            ], style={"marginTop": "4px", "lineHeight": "1.5"}),
            html.Div([
                html.Strong("Take forward: "), html.Span(takeaway),
            ], style={"marginTop": "4px", "lineHeight": "1.5"}),
        ],
        style={
            "backgroundColor": "#E9F5F5", "borderLeft": "5px solid #2A9D8F",
            "padding": "14px 18px", "borderRadius": "8px", "color": "#425466",
            "marginBottom": "16px",
        },
    )


# Build reusable learning cards containing concise evidence bullets.
def learning_card(title, bullets, accent="#165D7A"):
    return html.Div(
        [
            html.Div(title, style={
                "fontSize": "12px", "fontWeight": "800", "letterSpacing": "1px",
                "textTransform": "uppercase", "color": accent,
            }),
            html.Ul(
                [html.Li(item) for item in bullets],
                style={"margin": "7px 0 0", "paddingLeft": "20px", "lineHeight": "1.55", "color": "#425466"},
            ),
        ],
        style={**CARD_STYLE, "borderLeft": f"5px solid {accent}", "padding": "14px 18px", "marginBottom": "16px"},
    )


# Build previous/next controls for the analytical journey.
def navigation_controls(previous_id, next_id, previous_label, next_label, previous_disabled=False, next_disabled=False):
    return html.Div(
        [
            html.Button(
                previous_label, id=previous_id, n_clicks=0, disabled=previous_disabled,
                style={
                    "border": "1px solid #B8C7D9", "borderRadius": "7px", "backgroundColor": "white",
                    "color": "#425466", "fontWeight": "700", "padding": "10px 15px", "cursor": "pointer",
                },
            ),
            html.Button(
                next_label, id=next_id, n_clicks=0, disabled=next_disabled,
                style={
                    "border": "none", "borderRadius": "7px", "backgroundColor": "#165D7A",
                    "color": "white", "fontWeight": "750", "padding": "10px 15px", "cursor": "pointer",
                },
            ),
        ],
        style={"display": "flex", "justifyContent": "space-between", "gap": "12px", "marginTop": "2px"},
    )


# Display the user's position in the three-step storyboard.
def journey_progress(active_step="profitability"):
    steps = [
        ("profitability", "1", "Where is value created?"),
        ("satisfaction", "2", "What explains the experience?"),
        ("advocacy", "3", "What should management do?"),
    ]
    children = []
    for index, (value, number, label) in enumerate(steps):
        active = value == active_step
        children.append(
            html.Div(
                [
                    html.Div(number, style={
                        "width": "28px", "height": "28px", "borderRadius": "50%",
                        "display": "flex", "alignItems": "center", "justifyContent": "center",
                        "backgroundColor": "#2A9D8F" if active else "#D9E2EC",
                        "color": "white" if active else "#5E6C84", "fontWeight": "800",
                    }),
                    html.Div(label, style={
                        "fontSize": "12px", "fontWeight": "800" if active else "600",
                        "color": "#172B4D" if active else "#6B778C", "marginTop": "5px",
                    }),
                ],
                style={"flex": "1", "textAlign": "center", "position": "relative"},
            )
        )
        if index < len(steps) - 1:
            children.append(html.Div(style={"height": "2px", "backgroundColor": "#D9E2EC", "flex": "0.55", "marginTop": "14px"}))
    return html.Div(
        [
            html.Div("Analytical journey", style={
                "fontSize": "11px", "fontWeight": "800", "letterSpacing": "1px",
                "textTransform": "uppercase", "color": "#0F6B78", "marginBottom": "10px",
            }),
            html.Div(children, style={"display": "flex", "alignItems": "flex-start"}),
        ],
        style={**CARD_STYLE, "padding": "13px 18px", "marginBottom": "14px"},
    )


# Update the learning cards from the filtered record- and client-level data.
def dynamic_learning_cards(filtered_rows, client_rows):
    if filtered_rows.empty:
        empty = learning_card("What did we learn?", ["No records match the current filters."], "#E76F51")
        return empty, empty, empty

    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    costs = filtered_rows[["HARDWARE", "SOFTWARE", "MANPOWER"]].sum().sort_values(ascending=False)
    strongest_service = filtered_rows[SERVICE_COLUMNS].mean().idxmax()
    weakest_service = filtered_rows[SERVICE_COLUMNS].mean().idxmin()
    service_priority = service_priority_data(filtered_rows, SERVICE_COLUMNS)
    if service_priority.empty:
        association_text = "No service–NPS association can be estimated for this selection."
    else:
        strongest_association = service_priority.loc[service_priority["Importance"].abs().idxmax()]
        association_text = (
            f"{strongest_association['Service Dimension']} has the largest observed absolute "
            f"Spearman association with NPS ({strongest_association['Importance']:+.2f}); this is descriptive, not causal."
        )
    nps_type = type_summary.loc[type_summary["Average_NPS"].idxmax()]

    if client_rows.empty:
        advocacy_bullets = ["No client-level advocacy group is available for the current selection."]
    else:
        advocacy_profit = client_rows.groupby("NPS Category", observed=True)["Gross_Profit"].sum().sort_values(ascending=False)
        leading_group = str(advocacy_profit.index[0])
        priority_count = int(client_rows["Priority Group"].eq("Prioritise").sum())
        advocacy_bullets = [
            f"{leading_group}s hold the largest client-level gross-profit pool (${advocacy_profit.iloc[0] / 1_000_000:.1f}M).",
            f"{priority_count:,} client(s) fall in the relative Prioritise zone of the client matrix.",
            "Use the client-level chart to differentiate protection, improvement and service-recovery actions.",
        ]

    profit_card = learning_card("What did we learn from financial value?", [
        f"{profit_leader['Profile']} leads the selected client-type comparison with ${profit_leader['Gross_Profit'] / 1_000_000:.1f}M gross profit.",
        f"{costs.index[0].title()} is the largest aggregate cost component in the filtered records.",
        "The next step is to test whether commercially important value is supported by a healthy client experience.",
    ], "#165D7A")
    satisfaction_card = learning_card("What did we learn from the experience?", [
        f"{SERVICE_LABELS[strongest_service]} has the highest average service rating ({filtered_rows[strongest_service].mean():.2f}/5).",
        f"{SERVICE_LABELS[weakest_service]} has the lowest average service rating ({filtered_rows[weakest_service].mean():.2f}/5).",
        f"{nps_type['Profile']} has the highest average NPS rating ({nps_type['Average_NPS']:.2f}/10) among client types.",
        association_text,
    ], "#2A9D8F")
    advocacy_card = learning_card("What did we learn for management action?", advocacy_bullets, "#E6A700")
    return profit_card, satisfaction_card, advocacy_card


# Create context-aware handoff messages between analytical sections.
def dynamic_handoffs(filtered_rows, client_rows):
    if filtered_rows.empty:
        message = "No matching records are available. Adjust the filters before continuing the journey."
        return transition_card("Continue the journey", message), transition_card("Continue the journey", message), transition_card("Final decision", message)
    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    weakest_service = filtered_rows[SERVICE_COLUMNS].mean().idxmin()
    if client_rows.empty:
        advocacy_text = "The final step is unavailable for this selection because no client-level advocacy groups remain."
    else:
        advocacy_text = "The final step converts the combined financial, experience and advocacy evidence into account-level action zones."
    return (
        transition_card("Next: test whether value is sustainable", f"{profit_leader['Profile']} is the current financial anchor (${profit_leader['Gross_Profit'] / 1_000_000:.1f}M gross profit). Now test whether the experience supports that value before recommending action."),
        transition_card("Next: connect experience to value", f"{SERVICE_LABELS[weakest_service]} is the weakest observed service dimension in the current selection. Compare this experience evidence with the commercial value carried by each advocacy group."),
        transition_card("Final decision step", advocacy_text),
    )


# Combine financial, service, and advocacy evidence into a final conclusion.
def integrated_conclusion(filtered_rows, client_rows):
    if filtered_rows.empty:
        return html.Div("No integrated conclusion can be calculated for an empty selection.", style={**CARD_STYLE, "padding": "20px"})
    type_summary = profile_summary(filtered_rows, "TYPE")
    profit_leader = type_summary.loc[type_summary["Gross_Profit"].idxmax()]
    service_means = filtered_rows[SERVICE_COLUMNS].mean()
    weakest_service = service_means.idxmin()
    strongest_service = service_means.idxmax()
    if client_rows.empty:
        advocacy_finding = "No client-level advocacy group remains under the current filters."
        action_finding = "Broaden the selection before applying account-level action zones."
    else:
        advocacy_profit = client_rows.groupby("NPS Category", observed=True)["Gross_Profit"].sum().sort_values(ascending=False)
        leading_group = str(advocacy_profit.index[0])
        priority_count = int(client_rows["Priority Group"].eq("Prioritise").sum())
        advocacy_finding = f"{leading_group}s carry the largest client-level gross-profit pool (${advocacy_profit.iloc[0] / 1_000_000:.1f}M)."
        action_finding = f"The client matrix identifies {priority_count:,} relative Prioritise account(s); inspect these alongside the Retain, Improve and Reconsider groups."
    return html.Div(
        [
            html.H3("Integrated conclusion", style={"margin": "0 0 10px", "color": "#172B4D"}),
            html.P(f"Financial finding: {profit_leader['Profile']} is the leading client-type value pool at ${profit_leader['Gross_Profit'] / 1_000_000:.1f}M cumulative gross profit.", style={"margin": "6px 0", "lineHeight": "1.55"}),
            html.P(f"Experience finding: {SERVICE_LABELS[strongest_service]} is the strongest observed service dimension, while {SERVICE_LABELS[weakest_service]} is the weakest in the current selection.", style={"margin": "6px 0", "lineHeight": "1.55"}),
            html.P(f"Advocacy finding: {advocacy_finding}", style={"margin": "6px 0", "lineHeight": "1.55"}),
            html.P(f"Management implication: protect high-value Promoters, improve commercially important Passives, and target high-value Detractors for service recovery where appropriate. {action_finding}", style={"margin": "6px 0", "lineHeight": "1.55"}),
            html.Div("Interpretation boundary: these are descriptive patterns and associations. They do not prove causation or predict churn, renewal or retention.", style={"fontSize": "12px", "color": "#6B778C", "marginTop": "12px", "fontStyle": "italic"}),
        ],
        style={**CARD_STYLE, "borderTop": "5px solid #2A9D8F", "padding": "20px 22px"},
    )

# Create the Dash application and configure its shared controls.
app = Dash(__name__, suppress_callback_exceptions=True)
app.title = "Client Value Storyboard"

segment_options = [
    {"label": label, "value": column}
    for column, label in SEGMENT_LABELS.items()
]

# Define the storyboard header and business question.
header = html.Div(
    [
        html.Div(
            "CLIENT VALUE STORYBOARD",
            style={
                "fontSize": "12px",
                "fontWeight": "800",
                "letterSpacing": "1.5px",
                "color": "#4DD0C8",
            },
        ),
        html.H1(
            "Where should the company protect, improve and grow client value?",
            style={
                "fontSize": "28px",
                "margin": "5px 0 8px",
                "color": "white",
            },
        ),
        html.Div(
            "Where is value created? → What explains the experience? → What should management do?",
            style={"color": "#D9E7F2", "fontSize": "15px"},
        ),
    ],
    style={
        "backgroundColor": "#172B4D",
        "padding": "23px 30px",
    },
)

# Define filters that apply across all three analytical sections.
global_filters = html.Div(
    [
        html.Div(
            [
                html.Label("Client type", style={"fontWeight": "700"}),
                dcc.Dropdown(
                    id="story-client-type",
                    options=[{"label": "All client types", "value": "All"}]
                    + [{"label": item, "value": item} for item in client_types],
                    value="All",
                    clearable=False,
                ),
            ]
        ),
        html.Div(
            [
                html.Label("Country", style={"fontWeight": "700"}),
                dcc.Dropdown(
                    id="story-country",
                    options=[{"label": "All countries", "value": "All"}]
                    + [{"label": item, "value": item} for item in countries],
                    value="All",
                    clearable=False,
                ),
            ]
        ),
        html.Div(
            [
                html.Label("NPS categories", style={"fontWeight": "700"}),
                dcc.Dropdown(
                    id="story-nps",
                    options=[{"label": item, "value": item} for item in NPS_ORDER],
                    value=NPS_ORDER,
                    multi=True,
                    clearable=False,
                ),
            ]
        ),
        html.Div(
            [
                html.Label("Observed years", style={"fontWeight": "700"}),
                dcc.RangeSlider(
                    id="story-years",
                    min=year_min,
                    max=year_max,
                    step=1,
                    value=[year_min, year_max],
                    marks={year: str(year) for year in years},
                    allowCross=False,
                ),
            ],
            style={"gridColumn": "span 2", "padding": "0 8px"},
        ),
        html.Button(
            "Reset filters",
            id="story-reset",
            n_clicks=0,
            style={
                "height": "38px",
                "border": "none",
                "borderRadius": "7px",
                "backgroundColor": "#2A9D8F",
                "color": "white",
                "fontWeight": "750",
                "cursor": "pointer",
            },
        ),
    ],
    style={
        **CARD_STYLE,
        "display": "grid",
        "gridTemplateColumns": "repeat(auto-fit, minmax(190px, 1fr))",
        "gap": "14px",
        "alignItems": "end",
        "padding": "16px 18px",
        "marginTop": "18px",
        "position": "sticky",
        "top": 0,
        "zIndex": 100,
    },
)

# Assemble the profitability section and its three figures.
profitability_tab = html.Div(
    [
        html.Div(
            [
                html.H2(
                    "1. Where is value created?",
                    style={"margin": 0, "color": "#172B4D"},
                ),
                html.P(
                    "Start by locating gross profit, checking whether it is supported by healthy margins, and identifying the costs that may weaken value.",
                    style={"color": "#5E6C84", "marginBottom": 0},
                ),
            ],
            style={"margin": "4px 0 15px"},
        ),
        purpose_card(
            "Which profiles create the greatest financial value?",
            "Establish the portfolio baseline before interpreting satisfaction or advocacy.",
            "High gross profit shows commercial importance, but not whether relationships are healthy.",
        ),
        html.Div(id="story-profit-learning"),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Profile breakdown", style={"fontWeight": "700"}),
                        dcc.Dropdown(
                            id="profit-profile",
                            options=segment_options,
                            value="TYPE",
                            clearable=False,
                        ),
                    ]
                ),
                html.Div(
                    [
                        html.Label("Financial metric", style={"fontWeight": "700"}),
                        dcc.Dropdown(
                            id="profit-metric",
                            options=[
                                {"label": "Gross profit", "value": "Gross_Profit"},
                                {"label": "Revenue", "value": "Revenue"},
                                {"label": "Client count", "value": "Clients"},
                            ],
                            value="Gross_Profit",
                            clearable=False,
                        ),
                    ]
                ),
                html.Div(
                    [
                        html.Label("Cost display", style={"fontWeight": "700"}),
                        dcc.RadioItems(
                            id="cost-mode",
                            options=[
                                {"label": "Stacked", "value": "stack"},
                                {"label": "Grouped", "value": "group"},
                            ],
                            value="stack",
                            inline=True,
                            labelStyle={"marginRight": "18px"},
                        ),
                    ]
                ),
            ],
            style={
                "display": "grid",
                "gridTemplateColumns": "repeat(auto-fit, minmax(260px, 1fr))",
                "gap": "15px",
                "marginBottom": "16px",
            },
        ),
        html.Div(
            [
                graph_card(
                    "profit-chart",
                    "Evidence 1 — Financial contribution",
                    "Which profiles create the most gross profit?",
                    "Use the metric selector to compare gross profit, revenue or client count. Bar length shows the selected measure; weighted gross-margin colour and hover details provide financial context.",
                    650,
                ),
                graph_card(
                    "cost-chart",
                    "Evidence 2 — Cost pressure",
                    "Which costs are absorbing value within each profile?",
                    "Compare hardware, software and manpower. A large cost total is not automatically a problem; interpret it together with the profile's gross profit and weighted margin.",
                    650,
                ),
                graph_card(
                    "value-experience-chart",
                    "Evidence 3 — Profit–satisfaction bridge",
                    "Which profiles combine healthy margins with strong service satisfaction?",
                    "Profiles toward the upper-right combine stronger satisfaction and weighted margins. Bubble size represents revenue and colour represents gross profit, providing the bridge to the satisfaction evidence.",
                    640,
                ),
                html.Div(id="story-profit-handoff"),
                navigation_controls("story-prev-profit", "story-next-profit", "Previous", "Continue to experience →", previous_disabled=True),
            ],
            style={"display": "grid", "gap": "18px"},
        ),
    ],
    style={"padding": "20px 0"},
)

# Assemble the satisfaction and loyalty section.
satisfaction_tab = html.Div(
    [
        html.Div(
            [
                html.H2(
                    "2. What explains the experience?",
                    style={"margin": 0, "color": "#172B4D"},
                ),
                html.P(
                    "Compare current service performance, follow customer advocacy over time, then combine performance and NPS association into an action matrix.",
                    style={"color": "#5E6C84", "marginBottom": 0},
                ),
            ],
            style={"margin": "4px 0 15px"},
        ),
        purpose_card(
            "What do clients experience across service dimensions and periods?",
            "Examine strengths, gaps and observed NPS patterns before translating them into commercial priorities.",
            "Service associations guide investigation; they do not establish causal drivers.",
        ),
        html.Div(id="story-satisfaction-learning"),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Service-rating period", style={"fontWeight": "700"}),
                        dcc.Dropdown(
                            id="service-period",
                            options=[{"label": "Filtered period overall", "value": "Overall"}]
                            + [{"label": str(year), "value": str(year)} for year in years],
                            value="Overall", clearable=False,
                        ),
                    ]
                ),
                html.Div(
                    [
                        html.Label("Service dimensions (select 2+ for matrix)", style={"fontWeight": "700"}),
                        dcc.Dropdown(
                            id="story-services",
                            options=[{"label": SERVICE_LABELS[column], "value": column} for column in SERVICE_COLUMNS],
                            value=SERVICE_COLUMNS, multi=True, clearable=False,
                        ),
                    ]
                ),
            ],
            style={"display": "grid", "gridTemplateColumns": "repeat(auto-fit, minmax(280px, 1fr))", "gap": "15px", "marginBottom": "16px"},
        ),
        html.Div(
            [
                graph_card(
                    "service-chart",
                    "Evidence 4 — Service performance",
                    "Where are service strengths and gaps?",
                    "Compare all selected service dimensions on the same 1–5 scale. High ratings are strengths to protect; low ratings identify gaps, but record counts should be checked before acting.",
                    620,
                ),
                graph_card(
                    "nps-chart",
                    "Evidence 5 — Customer advocacy",
                    "Which client types show the strongest customer advocacy?",
                    "The heatmap shows average individual NPS ratings, not the formal Net Promoter Score. Read the printed values with the colours because the 7–9 colour scale magnifies small differences.",
                    590,
                ),
                graph_card(
                    "service-priority-chart",
                    "Evidence 6 — Service action",
                    "What should be improved or protected first?",
                    "Performance is the average service rating; importance is its Spearman association with NPS. The median lines create relative action quadrants, not universal pass–fail thresholds.",
                    650,
                ),
                html.Div(id="story-satisfaction-handoff"),
                navigation_controls("story-prev-satisfaction", "story-next-satisfaction", "← Back to value", "Continue to management action →"),
            ],
            style={"display": "grid", "gap": "18px"},
        ),
    ],
    style={"padding": "20px 0"},
)

# Assemble the advocacy and client-prioritisation section.
advocacy_tab = html.Div(
    [
        html.Div(
            [
                html.H2(
                    "3. What should management do?",
                    style={"margin": 0, "color": "#172B4D"},
                ),
                html.P(
                    "Aggregate the filtered years to one record per client, compare commercial value by advocacy category, and finish with the client-level action matrix.",
                    style={"color": "#5E6C84", "marginBottom": 0},
                ),
            ],
            style={"margin": "4px 0 15px"},
        ),
        purpose_card(
            "Which relationships deserve protection, improvement or service recovery?",
            "Combine gross profit, satisfaction and client-level advocacy to move from segment patterns to differentiated action.",
            "Use the matrix as a screening tool, not as a validated churn or retention model.",
        ),
        html.Div(id="story-advocacy-learning"),
        html.Div(
            [
                html.Label("Advocacy profile breakdown", style={"fontWeight": "700"}),
                dcc.Dropdown(
                    id="advocacy-profile",
                    options=segment_options,
                    value="TYPE",
                    clearable=False,
                ),
            ],
            style={"maxWidth": "420px", "marginBottom": "16px"},
        ),
        html.Div(
            [
                graph_card(
                    "advocacy-profit-chart",
                    "Evidence 7 — Advocacy value",
                    "Where is commercial value concentrated across advocacy groups?",
                    "High-value Passives are conversion opportunities; high-value Detractors may require service recovery; high-value Promoters are relationships to protect and grow.",
                    680,
                ),
                graph_card(
                    "advocacy-composition-chart",
                    "Evidence 8 — Portfolio mix",
                    "Which profiles contain Promoters, Passives and Detractors?",
                    "Stacked-bar widths show within-profile client percentages, while labels show absolute gross profit. Compare both measures because they use different denominators.",
                    680,
                ),
                html.Div(id="story-advocacy-handoff"),
                graph_card(
                    "client-priority-chart",
                    "Evidence 9 — Client action matrix",
                    "Which clients should be retained, improved or reconsidered?",
                    "The final chart combines client-level satisfaction, gross margin, gross profit and advocacy. Use the median zones as a screening tool, then inspect each client's hover details before recommending action.",
                    720,
                ),
                html.Div(id="story-conclusion"),
                navigation_controls("story-prev-advocacy", "story-next-advocacy", "← Back to experience", "End of journey", next_disabled=True),
            ],
            style={"display": "grid", "gap": "18px"},
        ),
    ],
    style={"padding": "20px 0"},
)

# Combine the header, filters, progress indicators, and three tabs.
app.layout = html.Div(
    [
        header,
        html.Div(
            [
                global_filters,
                html.Div(id="story-progress"),
                html.Div(
                    id="story-filter-summary",
                    style={
                        "fontSize": "12px",
                        "color": "#5E6C84",
                        "margin": "9px 2px 12px",
                    },
                ),
                html.Div(
                    id="story-kpis",
                    style={
                        "display": "grid",
                        "gridTemplateColumns": "repeat(auto-fit, minmax(210px, 1fr))",
                        "gap": "14px",
                        "marginBottom": "14px",
                    },
                ),
                html.Div(id="story-evidence-summary", style={"marginBottom": "18px"}),
                dcc.Tabs(
                    id="story-tabs",
                    value="profitability",
                    children=[
                        dcc.Tab(
                            label="1. Where is value created?",
                            value="profitability",
                            children=profitability_tab,
                        ),
                        dcc.Tab(
                            label="2. What explains the experience?",
                            value="satisfaction",
                            children=satisfaction_tab,
                        ),
                        dcc.Tab(
                            label="3. What should management do?",
                            value="advocacy",
                            children=advocacy_tab,
                        ),
                    ],
                ),
            ],
            style=CONTENT_STYLE,
        ),
        html.Footer(
            (
                "Descriptive decision support: average NPS rating is not the "
                "formal Net Promoter Score; association does not prove "
                "causation; observed longevity is not confirmed retention."
            ),
            style={
                "padding": "15px 28px",
                "fontSize": "12px",
                "textAlign": "center",
                "color": "#6B778C",
                "backgroundColor": "white",
                "borderTop": "1px solid #E3E8EF",
            },
        ),
    ],
    style=PAGE_STYLE,
)


In [7]:
# 7. Dashboard callbacks
# Connect filters and navigation to the figures, cards, and story state.

# Refresh all dashboard content when a filter or tab changes.
@app.callback(
    Output("story-progress", "children"),
    Output("story-kpis", "children"),
    Output("story-filter-summary", "children"),
    Output("story-evidence-summary", "children"),
    Output("profit-chart", "figure"),
    Output("cost-chart", "figure"),
    Output("value-experience-chart", "figure"),
    Output("service-chart", "figure"),
    Output("nps-chart", "figure"),
    Output("service-priority-chart", "figure"),
    Output("advocacy-profit-chart", "figure"),
    Output("advocacy-composition-chart", "figure"),
    Output("client-priority-chart", "figure"),
    Output("story-conclusion", "children"),
    Output("story-profit-learning", "children"),
    Output("story-satisfaction-learning", "children"),
    Output("story-advocacy-learning", "children"),
    Output("story-profit-handoff", "children"),
    Output("story-satisfaction-handoff", "children"),
    Output("story-advocacy-handoff", "children"),
    Input("story-tabs", "value"),
    Input("story-years", "value"),
    Input("story-client-type", "value"),
    Input("story-country", "value"),
    Input("story-nps", "value"),
    Input("story-services", "value"),
    Input("profit-profile", "value"),
    Input("profit-metric", "value"),
    Input("cost-mode", "value"),
    Input("service-period", "value"),
    Input("advocacy-profile", "value"),
)
def update_storyboard(
    active_step,
    year_range,
    client_type,
    country,
    nps_categories,
    selected_services,
    profit_profile,
    profit_metric,
    cost_mode,
    service_period,
    advocacy_profile,
):
    nps_categories = nps_categories or []
    selected_services = selected_services or []

    filtered_rows = filter_rows(
        year_range,
        client_type,
        country,
        nps_categories,
    )

    # Client-level advocacy categories are assigned after aggregating the
    # selected years, so first retain all client-year NPS categories.
    client_source = filter_rows(
        year_range,
        client_type,
        country,
        NPS_ORDER,
    )
    client_rows = build_client_level(client_source)
    if nps_categories:
        client_rows = client_rows[
            client_rows["NPS Category"].astype(str).isin(nps_categories)
        ].copy()
    else:
        client_rows = client_rows.iloc[0:0].copy()

    segment_label = (
        "Overall"
        if client_type == "All"
        else client_type
    )
    client_profile = CLIENT_PROFILE_MAP[advocacy_profile]

    return (
        journey_progress(active_step),
        kpi_cards(filtered_rows),
        "Showing: " + filter_description(
            year_range, client_type, country, nps_categories
        ),
        evidence_summary(filtered_rows),
        profit_figure(filtered_rows, profit_profile, profit_metric),
        cost_figure(filtered_rows, profit_profile, cost_mode),
        value_experience_figure(filtered_rows, profit_profile),
        service_figure(
            filtered_rows, selected_services, service_period
        ),
        nps_heatmap_figure(filtered_rows),
        service_priority_figure(
            filtered_rows, selected_services, segment_label
        ),
        advocacy_profit_figure(client_rows, client_profile),
        advocacy_composition_figure(client_rows, client_profile),
        client_priority_figure(client_rows),
        integrated_conclusion(filtered_rows, client_rows),
        *dynamic_learning_cards(filtered_rows, client_rows),
        *dynamic_handoffs(filtered_rows, client_rows),
    )


# Reset every control to the full-portfolio default selection.
@app.callback(
    Output("story-years", "value"),
    Output("story-client-type", "value"),
    Output("story-country", "value"),
    Output("story-nps", "value"),
    Output("story-services", "value"),
    Output("profit-profile", "value"),
    Output("profit-metric", "value"),
    Output("cost-mode", "value"),
    Output("service-period", "value"),
    Output("advocacy-profile", "value"),
    Input("story-reset", "n_clicks"),
    prevent_initial_call=True,
)
def reset_storyboard(_clicks):
    return (
        [year_min, year_max],
        "All",
        "All",
        NPS_ORDER,
        SERVICE_COLUMNS,
        "TYPE",
        "Gross_Profit",
        "stack",
        "Overall",
        "TYPE",
    )




# Move between the three storyboard sections with the navigation buttons.
@app.callback(
    Output("story-tabs", "value"),
    Input("story-prev-profit", "n_clicks"),
    Input("story-next-profit", "n_clicks"),
    Input("story-prev-satisfaction", "n_clicks"),
    Input("story-next-satisfaction", "n_clicks"),
    Input("story-prev-advocacy", "n_clicks"),
    Input("story-next-advocacy", "n_clicks"),
    State("story-tabs", "value"),
    prevent_initial_call=True,
)
def navigate_story(_prev_profit, _next_profit, _prev_satisfaction, _next_satisfaction, _prev_advocacy, _next_advocacy, current_step):
    triggered = callback_context.triggered_id
    targets = {
        "story-next-profit": "satisfaction",
        "story-prev-satisfaction": "profitability",
        "story-next-satisfaction": "advocacy",
        "story-prev-advocacy": "satisfaction",
    }
    return targets.get(triggered, current_step)


# Build the default figures once as a smoke check without starting a server.
# Build representative default figures as a smoke test without starting Dash.
_default_rows = filter_rows(
    [year_min, year_max], "All", "All", NPS_ORDER
)
_default_clients = build_client_level(_default_rows)
_smoke_figures = [
    profit_figure(_default_rows, "TYPE", "Gross_Profit"),
    profit_figure(_default_rows, "TYPE", "Revenue"),
    profit_figure(_default_rows, "TYPE", "Clients"),
    cost_figure(_default_rows, "TYPE", "stack"),
    value_experience_figure(_default_rows, "TYPE"),
    service_figure(_default_rows, SERVICE_COLUMNS, "Overall"),
    nps_heatmap_figure(_default_rows),
    service_priority_figure(_default_rows, SERVICE_COLUMNS, "Overall"),
    advocacy_profit_figure(_default_clients, "Client_Type"),
    advocacy_composition_figure(_default_clients, "Client_Type"),
    client_priority_figure(_default_clients),
]
assert all(isinstance(figure, go.Figure) for figure in _smoke_figures)
print("Combined storyboard ready: 3 navigation sections, 9 dashboard figures, story progress, learning cards, handoffs, navigation controls and an integrated conclusion.")


Combined storyboard ready: 3 navigation sections, 9 dashboard figures, story progress, learning cards, handoffs, navigation controls and an integrated conclusion.


## Open the combined storyboard in a browser

1. Run every code cell above in order.
2. Start the Dash server manually only when presenting by uncommenting the launch command in the final code cell.
3. The dashboard uses three tabs: **Where is value created?**, **What explains the experience?**, and **What should management do?**
4. Global filters apply across the story. Section-specific controls refine the relevant charts.
5. The year filter selects observed-year overlap; client-level financial totals are cumulative across the included observations, not annualised.


In [8]:
# Presentation launch command — uncomment and run manually after notebook execution.
# app.run(debug=False, jupyter_mode="external", port=8052)


## Interpretation safeguards

- Gross profit = revenue − hardware − software − manpower.
- Client-year figures remain record-level; advocacy and priority figures aggregate to one row per client.
- NPS composition percentages are within-profile client shares; gross-profit labels are absolute profile totals.
- Median action zones are relative screening categories, not universal thresholds.
- The dashboard reports descriptive associations only. It does not prove causation or predict churn, cancellation, renewal or retention.
- Observed longevity describes historical record span and is not a retention probability.
